<a href="https://colab.research.google.com/github/arnab326/FeatureExtraction-/blob/main/featurelens_experiments_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os, sys, json, random, subprocess


In [2]:
INSTALL = True
if INSTALL:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "transformers", "datasets", "scikit-learn", "captum",
                    "spacy", "scipy", "matplotlib"], check=True)
    subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm"], check=True)

In [3]:
import numpy as np
import torch
from datasets import load_dataset
from scipy.stats import spearmanr
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import chi2
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                             recall_score, confusion_matrix)
from transformers import (AutoTokenizer, AutoModel,
                          AutoModelForSequenceClassification)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

Device: cuda


In [4]:
DATASET = "fancyzhx/ag_news"   # or "tweet_eval_offensive" for the real-world audit
CFG = {
    "fancyzhx/ag_news": {
        "load": lambda: load_dataset("fancyzhx/ag_news"),
        "text": "text", "label": "label", "test_split": "test",
        "class_names": ["World", "Sports", "Business", "Sci/Tech"],
    },
    "tweet_eval_offensive": {
        "load": lambda: load_dataset("tweet_eval", "offensive"),
        "text": "text", "label": "label", "test_split": "test",
        "class_names": ["not-offensive", "offensive"],
    },
}[DATASET]

N_TRAIN = 8000    # sentences used by BOTH the ML and DL model (fair comparison)
N_TEST  = 2000    # evaluation sentences
N_IG    = 300     # sentences explained with Integrated Gradients (IG is slow)
N_PROBE = 3000    # sentences for probing
MODEL_NAME = "distilbert-base-uncased"   # compact, fits a free T4
PROBE_BACKBONE = "bert-base-uncased"     # frozen, 12 layers + embeddings = 13 hidden states
OUT = "featurelens_results"
os.makedirs(OUT, exist_ok=True)
results = {"dataset": DATASET, "seed": SEED}

In [5]:
ds = CFG["load"]()
train = ds["train"].shuffle(seed=SEED).select(range(min(N_TRAIN, len(ds["train"]))))
test  = ds[CFG["test_split"]].shuffle(seed=SEED).select(range(min(N_TEST, len(ds[CFG["test_split"]]))))
Xtr, ytr = list(train[CFG["text"]]), np.array(train[CFG["label"]])
Xte, yte = list(test[CFG["text"]]),  np.array(test[CFG["label"]])

NC = int(max(ytr.max(), yte.max())) + 1            # <-- number of classes, READ FROM DATA
CLASS_NAMES = CFG["class_names"][:NC]
results["num_classes"] = NC
print(f"{len(Xtr)} train / {len(Xte)} test sentences | {NC} classes: {CLASS_NAMES}")
print("train class balance:", np.round(np.bincount(ytr) / len(ytr), 3))

README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 18.6MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.23MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

8000 train / 2000 test sentences | 4 classes: ['World', 'Sports', 'Business', 'Sci/Tech']
train class balance: [0.252 0.253 0.236 0.258]


In [6]:
vec = TfidfVectorizer(ngram_range=(1, 2), min_df=5, stop_words="english", sublinear_tf=True)
A_tr = vec.fit_transform(Xtr)
A_te = vec.transform(Xte)
ml_clf = LogisticRegression(max_iter=2000).fit(A_tr, ytr)
ml_pred = ml_clf.predict(A_te)
ml_acc = accuracy_score(yte, ml_pred)

In [7]:
uvec = TfidfVectorizer(min_df=5, stop_words="english", sublinear_tf=True)
U_tr = uvec.fit_transform(Xtr)
vocab = uvec.get_feature_names_out()

chi2_top = {}   # class name -> list of {word, chi2}
for c in range(NC):
    scores, _ = chi2(U_tr, (ytr == c).astype(int))   # one-vs-rest chi-square
    scores = np.nan_to_num(scores)
    order = np.argsort(-scores)[:30]
    chi2_top[CLASS_NAMES[c]] = [{"word": str(vocab[j]), "chi2": float(scores[j])} for j in order]

results["ml"] = {
    "accuracy": float(ml_acc),
    "macro_f1": float(f1_score(yte, ml_pred, average="macro")),
    "chi2_top_words": chi2_top,
}
print(f"ML accuracy (TF-IDF 1-2gram + LR): {ml_acc:.3f}")
for c in range(NC):
    print(f"  chi2 top [{CLASS_NAMES[c]}]:",
          [d['word'] for d in chi2_top[CLASS_NAMES[c]][:8]])

ML accuracy (TF-IDF 1-2gram + LR): 0.889
  chi2 top [World]: ['iraq', 'killed', 'minister', 'baghdad', 'iraqi', 'palestinian', 'troops', 'iran']
  chi2 top [Sports]: ['season', 'cup', 'game', 'league', 'coach', 'win', 'team', 'victory']
  chi2 top [Business]: ['oil', 'stocks', 'prices', 'fullquote', 'profit', 'shares', 'percent', 'aspx']
  chi2 top [Sci/Tech]: ['microsoft', 'software', 'space', 'internet', 'linux', 'web', 'windows', 'users']


In [8]:
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=NC).to(DEVICE)

def batches(texts, labels, bs, shuffle=True):
    idx = np.random.permutation(len(texts)) if shuffle else np.arange(len(texts))
    for i in range(0, len(idx), bs):
        b = idx[i:i + bs]
        enc = tok([texts[j] for j in b], padding=True, truncation=True,
                  max_length=128, return_tensors="pt").to(DEVICE)
        yield enc, torch.tensor(labels[b]).to(DEVICE)

EPOCHS, BS = 2, 16
opt = torch.optim.AdamW(model.parameters(), lr=3e-5)
for ep in range(EPOCHS):
    model.train(); total = 0.0
    for enc, y in batches(Xtr, ytr, BS):
        loss = model(**enc, labels=y).loss
        loss.backward(); opt.step(); opt.zero_grad()
        total += loss.item()
    print(f"epoch {ep + 1}: mean train loss {total / (len(Xtr) / BS):.4f}")

@torch.no_grad()
def predict(texts, bs=64):
    model.eval(); preds = []
    for i in range(0, len(texts), bs):
        enc = tok(texts[i:i + bs], padding=True, truncation=True,
                  max_length=128, return_tensors="pt").to(DEVICE)
        preds += model(**enc).logits.argmax(-1).tolist()
    return np.array(preds)

dl_pred = predict(Xte)
dl_acc = accuracy_score(yte, dl_pred)
results["dl"] = {
    "accuracy": float(dl_acc),
    "macro_f1": float(f1_score(yte, dl_pred, average="macro")),
    "precision_macro": float(precision_score(yte, dl_pred, average="macro", zero_division=0)),
    "recall_macro": float(recall_score(yte, dl_pred, average="macro", zero_division=0)),
    "confusion_matrix": confusion_matrix(yte, dl_pred).tolist(),
    "class_names": CLASS_NAMES,
}
print(f"DL accuracy (DistilBERT): {dl_acc:.3f} | macro-F1 {results['dl']['macro_f1']:.3f}")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


epoch 1: mean train loss 0.3865
epoch 2: mean train loss 0.2014
DL accuracy (DistilBERT): 0.903 | macro-F1 0.904


In [9]:
from captum.attr import LayerIntegratedGradients

ref_token_id = tok.pad_token_id
lig = LayerIntegratedGradients(
    lambda ids, mask: model(input_ids=ids, attention_mask=mask).logits,
    model.get_input_embeddings())

def ig_word_scores(text, target):
    enc = tok(text, return_tensors="pt", truncation=True, max_length=128).to(DEVICE)
    ids, mask = enc["input_ids"], enc["attention_mask"]
    ref = torch.full_like(ids, ref_token_id); ref[0, 0] = ids[0, 0]; ref[0, -1] = ids[0, -1]
    att = lig.attribute(ids, baselines=ref, additional_forward_args=(mask,),
                        target=int(target), n_steps=32)
    att = att.sum(dim=-1).squeeze(0)                 # per-token attribution
    toks = tok.convert_ids_to_tokens(ids.squeeze(0))
    out = {}
    for t, a in zip(toks, att.tolist()):
        if t in tok.all_special_tokens:
            continue
        w = t.lstrip("##")
        out[w] = out.get(w, 0.0) + abs(a)            # fold word-pieces into words
    return out

In [10]:
ig_per_class = {c: {} for c in range(NC)}
sub = min(N_IG, len(Xte))
for i in range(sub):
    c = int(dl_pred[i])
    for w, a in ig_word_scores(Xte[i], c).items():
        ig_per_class[c][w] = ig_per_class[c].get(w, 0.0) + a
    if (i + 1) % 50 == 0:
        print(f"  IG {i + 1}/{sub}")

ig_top = {CLASS_NAMES[c]: [{"word": w, "ig": float(s)} for w, s in
          sorted(ig_per_class[c].items(), key=lambda kv: -kv[1])[:30]] for c in range(NC)}
results["dl"]["ig_top_words"] = ig_top

  IG 50/300
  IG 100/300
  IG 150/300
  IG 200/300
  IG 250/300
  IG 300/300


In [11]:
def topk_overlap(a, b, k=20):
    sa, sb = set(a[:k]), set(b[:k])
    return len(sa & sb) / k

bridge = {}
for c in range(NC):
    cn = CLASS_NAMES[c]
    chi_words = [d["word"] for d in chi2_top[cn]]
    ig_words  = [d["word"] for d in ig_top[cn]]
    overlap = topk_overlap(chi_words, ig_words, k=20)
    # Spearman on the shared vocabulary (rank agreement where both methods have a score)
    chi_rank = {w: r for r, w in enumerate(chi_words)}
    ig_rank  = {w: r for r, w in enumerate(ig_words)}
    shared = [w for w in chi_rank if w in ig_rank]
    rho = float(spearmanr([chi_rank[w] for w in shared],
                          [ig_rank[w] for w in shared]).correlation) if len(shared) >= 3 else None
    bridge[cn] = {"top20_overlap": overlap, "spearman_shared": rho, "n_shared": len(shared)}
    print(f"  bridge [{cn}]: overlap@20={overlap:.2f}, shared={len(shared)}, spearman={rho}")
results["bridge"] = bridge

  bridge [World]: overlap@20=0.15, shared=8, spearman=-0.3333333333333334
  bridge [Sports]: overlap@20=0.40, shared=9, spearman=0.31666666666666665
  bridge [Business]: overlap@20=0.30, shared=7, spearman=0.5357142857142858
  bridge [Sci/Tech]: overlap@20=0.35, shared=8, spearman=0.4047619047619048


In [12]:
import spacy
nlp = spacy.load("en_core_web_sm", disable=["ner", "lemmatizer"])

ptexts = Xtr[:N_PROBE]
plabels_topic = ytr[:N_PROBE]

def tree_depth(doc):
    def d(tok): return 1 + max((d(c) for c in tok.children), default=0)
    return max((d(s.root) for s in doc.sents), default=1)

lengths, depths = [], []
for doc in nlp.pipe(ptexts, batch_size=64):
    lengths.append(len(doc)); depths.append(tree_depth(doc))
lengths, depths = np.array(lengths), np.array(depths)

def to_bins(x, n=3):   # quantile bins -> a balanced classification target
    edges = np.quantile(x, np.linspace(0, 1, n + 1)[1:-1])
    return np.digitize(x, edges)

probe_targets = {
    "surface_length": to_bins(lengths, 3),
    "syntax_treedepth": to_bins(depths, 3),
    "semantic_topic": plabels_topic,
}

ptok = AutoTokenizer.from_pretrained(PROBE_BACKBONE)
pbert = AutoModel.from_pretrained(PROBE_BACKBONE, output_hidden_states=True).to(DEVICE).eval()

@torch.no_grad()
def layer_reps(texts, bs=64):
    """Return list of 13 arrays (one per layer) of mean-pooled hidden states."""
    acc = None
    for i in range(0, len(texts), bs):
        enc = ptok(texts[i:i + bs], padding=True, truncation=True,
                   max_length=128, return_tensors="pt").to(DEVICE)
        hs = pbert(**enc).hidden_states                 # tuple of 13 tensors
        m = enc["attention_mask"].unsqueeze(-1)
        pooled = [((h * m).sum(1) / m.sum(1)).cpu().numpy() for h in hs]
        acc = pooled if acc is None else [np.vstack([a, p]) for a, p in zip(acc, pooled)]
    return acc

reps = layer_reps(ptexts)
n_layers = len(reps)
split = int(0.8 * N_PROBE)

probe = {}
for name, y in probe_targets.items():
    majority = float(np.bincount(y).max() / len(y))
    accs = []
    for L in range(n_layers):
        Z = reps[L]
        clf = LogisticRegression(max_iter=1000).fit(Z[:split], y[:split])
        accs.append(float(clf.score(Z[split:], y[split:])))
    probe[name] = {"layer_accuracy": accs, "majority_baseline": majority}
    print(f"  probe [{name}]: baseline={majority:.3f}, best={max(accs):.3f} @layer {int(np.argmax(accs))}")
results["probing"] = probe

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  probe [surface_length]: baseline=0.366, best=0.883 @layer 0
  probe [syntax_treedepth]: baseline=0.432, best=0.568 @layer 7
  probe [semantic_topic]: baseline=0.265, best=0.873 @layer 12


In [13]:
with open(os.path.join(OUT, "results.json"), "w") as f:
    json.dump(results, f, indent=2)

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.figure(figsize=(8, 4.5))
for name, d in probe.items():
    plt.plot(range(n_layers), d["layer_accuracy"], marker="o", label=name)
    plt.axhline(d["majority_baseline"], ls="--", alpha=0.3)
plt.xlabel("BERT layer (0 = embeddings)"); plt.ylabel("probe accuracy")
plt.title("What each layer of BERT makes linearly decodable")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(os.path.join(OUT, "probing_plot.png"), dpi=200)
print("Saved results.json + probing_plot.png to", OUT)

Saved results.json + probing_plot.png to featurelens_results


In [16]:
import os
from google.colab import files

# Zip the results folder
os.system("zip -r featurelens_results.zip featurelens_results")

# Download the zipped file
files.download("featurelens_results.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>